# Team Selous — Health QA: Retrieval-Assisted Fine-Tuning

TRI Saturday AI Course project. Fine-tunes **Qwen3-4B** (Unsloth, QLoRA) on a
free Colab T4 to answer short health questions, then decides — per validation
results — whether a plain retrieval baseline or the fine-tuned model gives the
better submission.

**To run:** fresh Colab GPU runtime (T4). Upload `train_qa.csv` when prompted,
then `test_questions.csv` at the submission step. Renamed uploads (e.g.
`train_qa(1).csv`) are accepted automatically.

**Structure:** model load → LoRA → data prep → training → validation → submission.
Training and validation repeat across three cross-validation folds, followed by
a full-data refit inside the submission stage.

## Method

- **Answer-only supervised loss.** The prompt (including few-shot examples) is
  masked out of the loss; only the answer tokens and end-of-turn token are
  supervised. No general-chat data mixture.
- **Retrieval-assisted few-shot prompting.** A word + character TF-IDF bank
  retrieves the two most similar training questions and includes them in the
  prompt as style examples. Retrieval always excludes the target's own
  document/duplicate group, both during training and validation.
- **Conservative LoRA settings:** rank 8, alpha 16, all seven attention/MLP
  projections, learning rate 5e-5, three epochs, effective batch size 4,
  FP16, 768-token context — deliberately conservative starting points given
  the dataset size, not a tuned optimum.
- **Deterministic decoding.** Greedy decoding, reasoning/thinking disabled,
  a mild 1.05 repetition penalty, and a generation length ceiling derived
  from the training answers (rather than an arbitrary cap).
- **Grouped, topic-stratified 3-fold cross-validation** compares plain
  retrieval, fine-tuned generation, and two confidence-gated hybrids, scored
  by raw character-level Levenshtein distance (the competition metric).
  Rows that share a document or an answer are kept in the same fold so the
  model is never validated on something too similar to its own training data.
- **Policy selection is automatic:** the fine-tuned or hybrid policy is only
  chosen over plain retrieval if it beats it by a meaningful margin across
  folds; otherwise the notebook falls back to retrieval for the submission.
- **Final fit on all 83 rows**, starting from the original (untrained) adapter
  — validation-fold adapters are never carried into the submission model.

Reference answers average ~70.5 characters (range 29–110); matching phrasing
and punctuation matters as much as brevity for this metric.

## Notes and limitations

- Local cross-validation gives an optimistic estimate of leaderboard
  performance — fold selection is not an unbiased test of the final score.
- The dataset mixes 43 document-linked questions and 40 additional questions
  without document provenance; the two groups have a noticeably different
  question style, which the validation report breaks out separately.
- This workflow validates against the competition metric only — it is not a
  clinical review of the reference answers or the model's outputs.


## 1. Installation and model load
Fresh runtime recommended so pip can resolve Unsloth's dependencies together.
Resolved package versions are saved with the run outputs for reproducibility.
A plain `transformers.Trainer` is used with explicit token labels rather than
a separate SFT trainer, since the loss mask (below) is custom.

In [ ]:
%pip install -q --upgrade unsloth scikit-learn pandas rapidfuzz

In [ ]:
# Import Unsloth before Transformers/PEFT.
from unsloth import FastLanguageModel
import torch
import gc, json, math, re, os, hashlib, subprocess, sys, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from transformers import Trainer, TrainingArguments, GenerationConfig, set_seed
from peft import get_peft_model_state_dict, set_peft_model_state_dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedGroupKFold
from rapidfuzz.distance import Levenshtein
from IPython.display import display

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
SEED = 3407
MODEL_NAME = 'unsloth/Qwen3-4B-unsloth-bnb-4bit'
MAX_SEQ_LENGTH = 768
EPOCHS = 3
N_FOLDS = 3
N_EXAMPLES = 2
LEARNING_RATE = 5e-5
# Set before running, not after seeing validation scores. None = automatic selection.
POLICY_OVERRIDE = None  # None, 'retrieval', 'generated', 'hybrid_065', 'hybrid_080'
RUN_DIR = Path('short_qa_run')
RUN_DIR.mkdir(exist_ok=True)
set_seed(SEED)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16, load_in_4bit=True, full_finetuning=False,
)
tokenizer.padding_side = 'right'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
TURN_END_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')
assert tokenizer.convert_ids_to_tokens(TURN_END_ID) == '<|im_end|>'
STOP_IDS = sorted(set([TURN_END_ID, tokenizer.eos_token_id]))
print(torch.cuda.get_device_name(0))
(RUN_DIR / 'requirements-resolved.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
)

## 2. LoRA adapter
Rank 8 with zero dropout — a small adaptation, appropriate for ~83 training
rows. An untouched copy of the initial adapter is kept so each cross-validation
fold and the final full-data fit all start from the same clean initialization.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED,
    use_rslora=False, loftq_config=None,
)
INITIAL_ADAPTER = {k: v.detach().cpu().clone()
                   for k, v in get_peft_model_state_dict(model).items()}
assert INITIAL_ADAPTER

def reset_adapter():
    FastLanguageModel.for_training(model)
    set_peft_model_state_dict(model, {k: v.clone() for k, v in INITIAL_ADAPTER.items()})
    actual = get_peft_model_state_dict(model)
    assert all(torch.equal(actual[k].detach().cpu(), v)
               for k, v in INITIAL_ADAPTER.items()), 'Adapter reset failed.'
    model.zero_grad(set_to_none=True)
    model.config.use_cache = False
    set_seed(SEED)

## 3. Data preparation
`QuestionId` and `document_id` are identifiers only, never model inputs.
Rows are grouped (same document, or a duplicate question/answer) so that
cross-validation folds never split near-identical examples across train and
validation. Groups are also stratified by topic.

In [ ]:
INPUT_COLUMNS = ['topic', 'care_setting', 'population', 'question']

def get_csv(preferred, required):
    path = Path(preferred)
    if not path.exists():
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError(f'Place {preferred} in the notebook directory.')
        print(f'Upload {preferred}')
        uploaded = files.upload()
        candidates = []
        for name in uploaded:
            if name.lower().endswith('.csv'):
                header = pd.read_csv(name, nrows=0).columns
                if set(required).issubset(header):
                    candidates.append(Path(name))
        if len(candidates) != 1:
            raise ValueError(f'Expected exactly one matching CSV, found {candidates}.')
        path = candidates[0]
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = set(required) - set(frame.columns)
    assert not missing, f'Missing columns: {missing}'
    assert len(frame), 'CSV is empty.'
    return frame, path

qa, train_path = get_csv('train_qa.csv', INPUT_COLUMNS + ['reference_answer', 'QuestionId'])
if 'document_id' not in qa:
    qa['document_id'] = ''
assert qa['QuestionId'].is_unique, 'Training QuestionId must be unique.'
assert qa['question'].str.strip().ne('').all()
assert qa['reference_answer'].str.strip().ne('').all()
assert qa['reference_answer'].map(lambda x: x == x.strip()).all(), 'Review whitespace in labels.'
print(f'{len(qa)} rows; {qa.document_id.str.strip().eq("").sum()} missing document IDs')
print('Answer character counts:')
display(qa.reference_answer.str.len().describe())

def norm(s):
    return re.sub(r'[^\w]+', ' ', unicodedata.normalize('NFKC', str(s)).casefold()).strip()

# Union-find handles transitive overlap between document and duplicate groups.
parent = list(range(len(qa)))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(i, j):
    parent[root(j)] = root(i)

seen = {}
for i, row in qa.iterrows():
    keys = [('question', norm(row.question)), ('answer', norm(row.reference_answer))]
    if row.document_id.strip():
        keys.append(('document', row.document_id.strip()))
    for key in keys:
        if key in seen:
            union(i, seen[key])
        else:
            seen[key] = i
qa['_group'] = [root(i) for i in range(len(qa))]
qa['_source'] = np.where(qa.document_id.str.strip().eq(''), 'missing_doc', 'known_doc')
splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(splitter.split(qa, y=qa.topic, groups=qa._group))
qa['_fold'] = -1
for f, (tr, va) in enumerate(folds):
    assert set(qa.iloc[tr]._group).isdisjoint(qa.iloc[va]._group)
    qa.loc[va, '_fold'] = f
    print(f'Fold {f}: train={len(tr)}, validation={len(va)}')
assert qa._fold.ge(0).all()
qa[['QuestionId', '_group', '_fold', '_source']].to_csv(RUN_DIR / 'folds.csv', index=False)
display(pd.crosstab(qa.topic, qa._fold))
display(pd.crosstab(qa._source, qa._fold))

### Retrieval bank and matching train/inference prompts
A word + character TF-IDF index over training questions retrieves the most
similar examples to use as in-prompt style guides (metadata match is a small
scoring bonus on top of question similarity). The same retrieval logic and
prompt format are used at both training and inference time. During training,
a row's own document/duplicate group is excluded from its own retrieval so
the model can't see its answer via the example bank.

In [ ]:
class QABank:
    def __init__(self, frame):
        self.df = frame.reset_index(drop=True).copy()
        questions = self.df.question.map(norm).tolist()
        self.word = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
        self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)
        self.w = self.word.fit_transform(questions)
        self.c = self.char.fit_transform(questions)

    def retrieve(self, row, exclude_group=None, k=N_EXAMPLES):
        q = norm(row['question'])
        lexical = (0.5 * (self.w @ self.word.transform([q]).T).toarray().ravel()
                   + 0.5 * (self.c @ self.char.transform([q]).T).toarray().ravel())
        score = 0.85 * lexical
        for col in ['topic', 'care_setting', 'population']:
            if str(row[col]).strip():
                score += 0.05 * self.df[col].eq(row[col]).to_numpy()
        allowed = np.ones(len(self.df), dtype=bool)
        if exclude_group is not None:
            allowed &= self.df['_group'].ne(exclude_group).to_numpy()
        ids = np.flatnonzero(allowed)
        assert len(ids), 'No eligible training examples remain.'
        order = ids[np.argsort(-score[ids], kind='stable')[:k]]
        return self.df.iloc[order], float(score[order[0]])

SYSTEM_PROMPT = (
    'Answer the health question with one brief clinical quick-reference answer. '
    'Use terse wording, usually about 8–15 words; use more only for essential details. '
    'Preserve necessary qualifiers, negation, and urgent actions. '
    'No preamble, question restatement, explanation, headings, bullets, or reasoning. '
    'Related examples show answer style and may help with wording; '
    'use their content only when it answers the actual question. '
    'Return only the answer, then stop.'
)

def user_content(row):
    return (f'Topic: {row["topic"]} | Care setting: {row["care_setting"]} | '
            f'Population: {row["population"]}\nQuestion: {row["question"]}')

def prompt_ids(row, examples):
    example_text = '\n\n'.join(
        user_content(ex) + '\nAnswer: ' + ex['reference_answer']
        for _, ex in examples.iterrows()
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Related examples:\n' + example_text
         + '\n\nNow answer this question:\n' + user_content(row)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, enable_thinking=False,
    )

def answer_ids(text):
    return tokenizer(text, add_special_tokens=False)['input_ids'] + [TURN_END_ID]

class AnswerDataset(torch.utils.data.Dataset):
    def __init__(self, frame, bank):
        self.items = []
        for _, row in frame.iterrows():
            examples, _ = bank.retrieve(row, exclude_group=row['_group'])
            prefix = prompt_ids(row, examples)
            target = answer_ids(row.reference_answer)
            assert not any(x in STOP_IDS for x in target[:-1]), 'Special token in answer.'
            assert len(prefix) + len(target) <= MAX_SEQ_LENGTH, 'Increase context; do not truncate targets.'
            self.items.append(dict(input_ids=prefix + target,
                                   attention_mask=[1] * (len(prefix) + len(target)),
                                   labels=[-100] * len(prefix) + target))
        assert self.items

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

# Explicit padding preserves supervised EOS even if pad_token_id == eos_token_id.
def answer_collator(items):
    width = math.ceil(max(len(x['input_ids']) for x in items) / 8) * 8
    return {
        key: torch.tensor([x[key] + [fill] * (width - len(x[key])) for x in items], dtype=torch.long)
        for key, fill in [('input_ids', tokenizer.pad_token_id), ('attention_mask', 0), ('labels', -100)]
    }

# This check runs in Colab; inspect that only the exact answer and EOS have labels.
probe_frame = qa.iloc[folds[0][0]]
probe = AnswerDataset(probe_frame, QABank(probe_frame))[0]
print('Supervised text:', tokenizer.decode([x for x in probe['labels'] if x != -100]))
print('Prompt tokens masked:', probe['labels'].count(-100))
del probe, probe_frame

## 4. Training
Each cross-validation fold trains from the same initial adapter with a fresh
optimizer, for a fixed 3 epochs (not tuned per fold, to avoid overfitting the
choice to only 83 rows). The training function is defined here and called
once per fold in the next section, then once more for the final full-data fit.

In [ ]:
def fit_adapter(frame, bank, name):
    reset_adapter()
    dataset = AnswerDataset(frame, bank)
    args = TrainingArguments(
        output_dir=str(RUN_DIR / name),
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
        warmup_ratio=0.1, lr_scheduler_type='linear',
        optim='adamw_8bit', weight_decay=0.01, max_grad_norm=1.0,
        fp16=True, bf16=False, logging_steps=5,
        save_strategy='no', eval_strategy='no', report_to='none',
        seed=SEED, data_seed=SEED, dataloader_num_workers=0,
        remove_unused_columns=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset,
                      data_collator=answer_collator, processing_class=tokenizer)
    result = trainer.train()
    print(name, result.metrics)
    trainer.optimizer = None
    trainer.lr_scheduler = None
    del trainer, dataset
    model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()

## 5. Validation evaluation
Every row gets a prediction from a fold adapter that never trained on its
group. Scoring uses raw Levenshtein distance on the untouched prediction and
reference strings — no normalization, lowercasing, or punctuation stripping,
matching the competition metric exactly.

Decoding is greedy and deterministic (not sampling) so results are
reproducible. The repetition penalty is kept mild (1.05) to avoid
suppressing wording that legitimately repeats between the question, the
few-shot examples, and the answer. The generation length ceiling is derived
from the longest training answer, not an arbitrary cutoff.

In [ ]:
def generation_limit(frame):
    return max(32, max(len(answer_ids(s)) for s in frame.reference_answer) + 8)

@torch.inference_mode()
def predict_generated(row, bank, token_limit):
    examples, confidence = bank.retrieve(row)
    ids = prompt_ids(row, examples)
    assert len(ids) + token_limit <= MAX_SEQ_LENGTH, 'Prompt too long; increase context.'
    inputs = torch.tensor([ids], dtype=torch.long, device='cuda')
    # Fresh config avoids inherited sampling flags/penalties in model defaults.
    config = GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=token_limit,
        repetition_penalty=1.05, no_repeat_ngram_size=0,
        eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )
    out = model.generate(input_ids=inputs, attention_mask=torch.ones_like(inputs),
                         generation_config=config)
    continuation = out[0, len(ids):].tolist()
    answer = tokenizer.decode(continuation, skip_special_tokens=True,
                              clean_up_tokenization_spaces=False).strip()
    cap_hit = len(continuation) >= token_limit and continuation[-1] not in STOP_IDS
    return answer, examples.iloc[0].reference_answer, confidence, cap_hit

POLICIES = ['retrieval', 'generated', 'hybrid_065', 'hybrid_080']
def policy_answer(name, generated, retrieved, confidence):
    if name == 'retrieval': return retrieved
    if name == 'generated': return generated
    threshold = {'hybrid_065': 0.65, 'hybrid_080': 0.80}[name]
    return retrieved if confidence >= threshold else generated

# TRAIN -> VALIDATE, repeated sequentially. No validation labels enter fit_adapter.
oof_rows = []
for fold, (tr, va) in enumerate(folds):
    fold_train = qa.iloc[tr].copy()
    bank = QABank(fold_train)
    fit_adapter(fold_train, bank, f'fold_{fold}')
    FastLanguageModel.for_inference(model)
    token_limit = generation_limit(fold_train)
    for idx in va:
        row = qa.iloc[idx]
        generated, retrieved, confidence, cap_hit = predict_generated(row, bank, token_limit)
        result = {'row_index': int(idx), 'QuestionId': row.QuestionId, 'fold': fold,
                  'topic': row.topic, 'source': row['_source'], 'question': row.question,
                  'reference': row.reference_answer, 'confidence': confidence,
                  'cap_hit': cap_hit, 'generated_chars': len(generated),
                  'reference_chars': len(row.reference_answer)}
        for name in POLICIES:
            pred = policy_answer(name, generated, retrieved, confidence)
            result[name] = pred
            result[name + '_distance'] = Levenshtein.distance(pred, row.reference_answer)
        oof_rows.append(result)
    # Incremental results survive interruption if downloaded from Colab's files pane.
    pd.DataFrame(oof_rows).to_csv(RUN_DIR / 'oof_predictions_partial.csv', index=False)
    del bank, fold_train

oof = pd.DataFrame(oof_rows).sort_values('row_index').reset_index(drop=True)
assert len(oof) == len(qa) and oof.QuestionId.is_unique
oof.to_csv(RUN_DIR / 'oof_predictions.csv', index=False)

In [ ]:
summary = pd.DataFrame([
    {'policy': p, 'mean_distance': oof[p + '_distance'].mean(),
     'median_distance': oof[p + '_distance'].median(),
     'mean_answer_chars': oof[p].str.len().mean(),
     'exact_matches': int(oof[p + '_distance'].eq(0).sum())}
    for p in POLICIES
]).sort_values('mean_distance')
fold_scores = oof.groupby('fold')[[p + '_distance' for p in POLICIES]].mean()
display(summary)
display(fold_scores)
print('Reference mean characters:', oof.reference_chars.mean())
print('Generation cap hits:', int(oof.cap_hit.sum()))
print('Empty generated answers:', int(oof.generated.eq('').sum()))
display(oof.groupby('source')[[p + '_distance' for p in POLICIES]].mean())
display(oof.nlargest(12, 'generated_distance')[
    ['question', 'reference', 'generated', 'retrieval', 'generated_distance', 'cap_hit']])

# Small predeclared policy set; no large decoding/length/threshold search.
# Require >=2 characters mean improvement AND wins on >=2/3 folds over retrieval.
means = {p: float(oof[p + '_distance'].mean()) for p in POLICIES}
eligible = ['retrieval']
for p in POLICIES[1:]:
    wins = int((fold_scores[p + '_distance'] < fold_scores.retrieval_distance).sum())
    if means[p] <= means['retrieval'] - 2.0 and wins >= 2:
        eligible.append(p)
selected_policy = min(eligible, key=lambda p: means[p])
if POLICY_OVERRIDE is not None:
    assert POLICY_OVERRIDE in POLICIES
    selected_policy = POLICY_OVERRIDE
print('Submission policy:', selected_policy)
print('OOF selection score (not an unbiased final test estimate):', means[selected_policy])
summary.to_csv(RUN_DIR / 'validation_summary.csv', index=False)
fold_scores.to_csv(RUN_DIR / 'validation_by_fold.csv')

# Descriptive paired, group-bootstrap interval for selected minus retrieval.
# Accounts for within-group correlation, not policy-selection bias or hidden-test shift.
group_ids = qa.iloc[oof.row_index.to_numpy()]['_group'].to_numpy()
difference = (oof[selected_policy + '_distance'] - oof.retrieval_distance).to_numpy()
blocks = [difference[group_ids == g] for g in np.unique(group_ids)]
rng = np.random.default_rng(SEED)
bootstrap = [np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))]).mean()
             for _ in range(1000)]
print('Descriptive 95% interval, selected minus retrieval:', np.quantile(bootstrap, [0.025, 0.975]))

Per-row errors matter as much as the average — two questions about the same
condition can call for different actions, so retrieval similarity is not a
calibrated confidence score. The hybrid thresholds (0.65 / 0.80) are
hypotheses tested on these folds, not clinical thresholds. Grouped folds are
likely a harder test than the hidden leaderboard set (which may contain
closer paraphrases), so this cross-validation score should be read as a
lower-bound sanity check, not a leaderboard prediction.

The 43 document-linked and 40 additional questions have a noticeably
different question style; the source-grouped breakdown above is there to
surface that gap rather than hide it. This workflow evaluates against the
competition metric only — it is not a clinical review of the reference
answers or model outputs.

## 6. Full-data refit and submission generation
The policy chosen by cross-validation is locked in before the test file is
loaded. If it selected a generation-based policy, the adapter is reset and
trained on all 83 rows for the same 3 epochs; the retrieval bank is rebuilt
on the full training set. Output follows the required submission schema
(`QuestionId, Answer`). A retrieval-only alternative and a full per-row audit
are also saved, to inspect before uploading to the leaderboard.

In [ ]:
full_bank = QABank(qa)
if selected_policy != 'retrieval':
    fit_adapter(qa, full_bank, 'full_data')
    FastLanguageModel.for_inference(model)
    model.save_pretrained(str(RUN_DIR / 'team_selous_qwen_lora'))
    tokenizer.save_pretrained(str(RUN_DIR / 'team_selous_qwen_lora'))
else:
    print('Retrieval selected; no full-data fine-tuning needed.')

# Persist the bank: the retrieval-assisted adapter alone is not the full predictor.
qa[INPUT_COLUMNS + ['reference_answer', 'QuestionId', 'document_id', '_group']].to_csv(
    RUN_DIR / 'retrieval_bank.csv', index=False)
config = dict(model_name=MODEL_NAME, seed=SEED, epochs=EPOCHS,
              learning_rate=LEARNING_RATE, lora_rank=8, lora_alpha=16,
              max_seq_length=MAX_SEQ_LENGTH, n_examples=N_EXAMPLES,
              policy=selected_policy, system_prompt=SYSTEM_PROMPT,
              repetition_penalty=1.05, do_sample=False, enable_thinking=False,
              max_new_tokens=generation_limit(qa), oof_means=means,
              train_sha256=hashlib.sha256(train_path.read_bytes()).hexdigest())
(RUN_DIR / 'run_config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False))

In [ ]:
test_df, test_path = get_csv('test_questions.csv', INPUT_COLUMNS + ['QuestionId'])
assert test_df.QuestionId.is_unique, 'Test QuestionId must be unique.'
assert test_df.question.str.strip().ne('').all()
# Ignore any additional columns; never consume a test answer column.
test_df = test_df[INPUT_COLUMNS + ['QuestionId']].copy()
rows = []
for _, row in test_df.iterrows():
    examples, confidence = full_bank.retrieve(row)
    retrieved = examples.iloc[0].reference_answer
    generated, cap_hit = '', False
    # Run generation consistently for all rows when the chosen policy uses it.
    if selected_policy != 'retrieval':
        generated, retrieved, confidence, cap_hit = predict_generated(
            row, full_bank, generation_limit(qa))
    answer = policy_answer(selected_policy, generated, retrieved, confidence)
    rows.append({'QuestionId': row.QuestionId, 'Answer': answer,
                 'retrieval': retrieved, 'generated': generated,
                 'confidence': confidence, 'cap_hit': cap_hit})
audit = pd.DataFrame(rows)
audit.to_csv(RUN_DIR / 'submission_audit.csv', index=False)
submission = audit[['QuestionId', 'Answer']].copy()
assert len(submission) == len(test_df)
assert submission.QuestionId.tolist() == test_df.QuestionId.tolist()
assert submission.Answer.str.strip().ne('').all(), 'Empty answer: inspect submission_audit.csv before submitting.'
submission.to_csv('submission.csv', index=False)
audit.to_csv(RUN_DIR / 'submission_audit.csv', index=False)
audit[['QuestionId', 'retrieval']].rename(columns={'retrieval': 'Answer'}).to_csv(
    'submission_retrieval.csv', index=False)
print('Saved submission.csv:', len(submission), 'rows; policy:', selected_policy)
print('Generated cap hits:', int(audit.cap_hit.sum()))
display(submission.head(10))

In [ ]:
# Download outputs; the ZIP includes metrics, environment, bank, config, and adapter if used.
import shutil
shutil.copy2('submission.csv', RUN_DIR / 'submission.csv')
shutil.copy2('submission_retrieval.csv', RUN_DIR / 'submission_retrieval.csv')
archive = shutil.make_archive('short_qa_outputs', 'zip', root_dir=RUN_DIR)
try:
    from google.colab import files
    files.download('submission.csv')
    files.download(archive)
except ImportError:
    print('Outputs:', Path('submission.csv').resolve(), Path(archive).resolve())

## References
- [Qwen3-4B model card](https://huggingface.co/Qwen/Qwen3-4B) — non-thinking
  chat template and vendor sampling recommendations (greedy decoding here is a
  task-specific choice, not the vendor default).
- [Unsloth Qwen3 guide](https://unsloth.ai/docs/models/tutorials/qwen3-how-to-run-and-fine-tune) —
  `enable_thinking=False` at prompt construction.
- [Transformers Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer) —
  explicit token labels and a custom data collator.
- [PEFT functional helpers](https://huggingface.co/docs/peft/en/package_reference/functional) —
  adapter state extraction/restoration between cross-validation folds.
- [Unsloth installation guide](https://unsloth.ai/docs/get-started/install/pip-install).

Adapted from Unsloth's example notebooks
([LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)).

**Team Selous** — TRI Saturday AI Course.
